In [1]:
import torch
import torch.nn as nn

# A standard feedforward network
ff_net = nn.Linear(in_features=1, out_features=1)

# Three temperature readings - treated as independent inputs
day1 = torch.tensor([[15.0]])
day2 = torch.tensor([[16.0]])
day3 = torch.tensor([[14.0]])

out1 = ff_net(day1)
out2 = ff_net(day2)
out3 = ff_net(day3)

print(f"Output for day1: {out1.item():.4f}")
print(f"Output for day2: {out2.item():.4f}")
print(f"Output for day3: {out3.item():.4f}")

#Reverse the order
out_a = ff_net(day3)
out_b = ff_net(day2)
out_c = ff_net(day1)

print(f"\nReversed order outputs: {out_a.item():.4f}, {out_b.item():.4f}, {out_c.item():.4f}")
print(f"\nSame outputs? {torch.allclose(out1, out_c)}")


Output for day1: 10.9779
Output for day2: 11.6569
Output for day3: 10.2989

Reversed order outputs: 10.2989, 11.6569, 10.9779

Same outputs? True


In [3]:
import pandas as pd
import urllib.request
import zipfile
import os

#Download directly from Keras hosted storage
URL = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip"
ZIP_PATH = "../data/jena_climate.zip"
CSV_PATH = "../data/jena_climate_2009_2016.csv"

os.makedirs("../data", exist_ok=True)

if not os.path.exists(CSV_PATH):
    print("Downloading dataset...")
    urllib.request.urlretrieve(URL, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall("../data")
    print("Done.")
else:
    print("Already downloaded.")

df = pd.read_csv(CSV_PATH)
print(df.shape)
print(df.columns.tolist())

Already downloaded.
(420551, 15)
['Date Time', 'p (mbar)', 'T (degC)', 'Tpot (K)', 'Tdew (degC)', 'rh (%)', 'VPmax (mbar)', 'VPact (mbar)', 'VPdef (mbar)', 'sh (g/kg)', 'H2OC (mmol/mol)', 'rho (g/m**3)', 'wv (m/s)', 'max. wv (m/s)', 'wd (deg)']


In [5]:
import plotly.graph_objects as go

temp = df["T (degC)"].values
time = df["Date Time"].values

print(f"Temperature range: {temp.min():.1f}°C to {temp.max():.1f}°C")
print(f"Mean: {temp.mean():.1f}°C")
print(f"Total timesteps: {len(temp)}")
print(f"Sampling interval: every 10 minutes")
print(f"Total duration: {len(temp) * 10 / 60 / 24 / 365:.1f} years")

# Plot one month of data so pattern is visible
one_month = 6 * 24 * 30

fig = go.Figure()
fig.add_trace(go.Scatter(
    y=temp[:one_month],
    mode="lines",
    line=dict(width=1, color="steelblue"),
    name="Temperature (degC)"
))

fig.update_layout(
    title="Jena Climate — first 30 days of temperature",
    xaxis_title="Timestep (every 10 min)",
    yaxis_title="Temperature (°C)",
    height=400
)
fig.show()



Temperature range: -23.0°C to 37.3°C
Mean: 9.5°C
Total timesteps: 420551
Sampling interval: every 10 minutes
Total duration: 8.0 years


In [6]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler

temp_data = df["T (degC)"].values.reshape(-1,1)

scaler = MinMaxScaler()
temp_scaled = scaler.fit_transform(temp_data).flatten()

#Sliding window - given SEQ_LEN steps, predict the next one
SEQ_LEN = 144 # one full day of history

def create_sequences(data: np.ndarray, seq_len: int):
    """
    Convert a 1D time series into (input_sequence, target) pairs.
    
    Args:
        data: normalized 1D array of temperature values
        seq_len: number of past timesteps to use as input
    
    Returns:
        X: shape (N, seq_len) — input sequences
        y: shape (N,)        — next-step targets
    """
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i: i+seq_len])
        y.append(data[i+seq_len])
    return np.array(X), np.array(y)

X, y = create_sequences(temp_scaled, SEQ_LEN)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nExample — first input sequence (first and last 3 values):")
print(f"  Start: {X[0, :3]}")
print(f"  End:   {X[0, -3:]}")
print(f"  Target (next step): {y[0]:.4f}")


X shape: (420407, 144)
y shape: (420407,)

Example — first input sequence (first and last 3 values):
  Start: [0.24863161 0.24216288 0.24050423]
  End:   [0.30983579 0.30568917 0.30635263]
  Target (next step): 0.3054


In [10]:
import torch
from torch.utils.data import DataLoader, TensorDataset

# Chronological split - never shuffle time series data
n = len(X)
n_train = int(n * 0.7)
n_val = int(n * 0.85)

X_train, y_train = X[:n_train], y[:n_train]
X_val, y_val = X[n_train: n_val], y[n_train: n_val]
X_test, y_test = X[n_val:], y[n_val:]

print(f"Train:      {X_train.shape[0]:>7,} samples")
print(f"Validation: {X_val.shape[0]:>7,} samples")
print(f"Test:       {X_test.shape[0]:>7,} samples")

# Convert to PyTorch tensors - RNN expects (batch, seq_len, input_size)
def make_loader(X: np.ndarray, y: np.ndarray, batch_size: int, shuffle: bool) -> DataLoader:
    """
    Wrap numpy arrays into a PyTorch DataLoader.
    
    Args:
        X: input sequences, shape (N, seq_len)
        y: targets, shape (N,)
        batch_size: samples per batch
        shuffle: True for train, False for val/test
    
    Returns:
        DataLoader yielding (X_batch, y_batch) tensors
    """

    X_t = torch.tensor(X, dtype=torch.float32).unsqueeze(-1)
    y_t = torch.tensor(y, dtype=torch.float32)
    dataset = TensorDataset(X_t, y_t)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

BATCH_SIZE = 256
train_loader = make_loader(X_train, y_train, BATCH_SIZE, shuffle=False)
val_loader   = make_loader(X_val,   y_val,   BATCH_SIZE, shuffle=False)
test_loader  = make_loader(X_test,  y_test,  BATCH_SIZE, shuffle=False)

# Verify one batch
X_batch, y_batch = next(iter(train_loader))
print(f"\nOne batch — X: {X_batch.shape}, y: {y_batch.shape}")

Train:      294,284 samples
Validation:  63,061 samples
Test:        63,062 samples

One batch — X: torch.Size([256, 144, 1]), y: torch.Size([256])
